# Sector Rotation System — Setup & Backtest

This notebook runs on **Google Colab** (free tier). It:

1. Clones the repo and installs dependencies
2. Populates the database with 2 years of historical data
3. Runs the full pipeline (regime → optimizer → screener → NLP)
4. Executes a walk-forward backtest and displays results
5. Shows the current Executive Summary with dollar allocations

**Runtime:** ~5–8 minutes on Colab free tier.

## 1. Clone Repo & Install Dependencies

In [11]:
# --- Clone private repo using Colab Secrets ---
# Store your GitHub PAT in Colab Secrets (key icon in sidebar) as GITHUB_TOKEN.
# The token needs 'repo' scope for private repository access.

import os
from pathlib import Path

REPO_DIR = '/content/sector-rotation-system'

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/bigbathtub101/sector-rotation-system.git {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install dependencies
!pip install -q -r requirements.txt
print('\nSetup complete.')

Repo already cloned at /content/sector-rotation-system, pulling latest...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 8 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.44 KiB | 191.00 KiB/s, done.
From https://github.com/bigbathtub101/sector-rotation-system
   592862a..569fe1d  main       -> origin/main
 * [new branch]      copilot/fix-data-ingestion-issues -> origin/copilot/fix-data-ingestion-issues
Updating 592862a..569fe1d
Fast-forward
 config.yaml                        |  4 ++--
 data_feeds.py                      | 25 +++++++++++++++++++++++--
 notebooks/setup_and_backtest.ipynb | 15 +++++++++++----
 3 files changed, 36 insertions(+), 8 deletions(-)
Working directory: /content/sector-rotation-system

Setup complete.


## 2. Set API Keys (Optional)

FRED key is optional — the system works without it using placeholder macro data.
Set it here if you have one.

In [12]:
import os

# Optional: set your FRED API key via Colab Secrets (free at https://fred.stlouisfed.org)
# Store your key in Colab Secrets (key icon in sidebar) as FRED_API_KEY.
try:
    from google.colab import userdata
    fred_key = userdata.get("FRED_API_KEY")
    if fred_key:
        os.environ["FRED_API_KEY"] = fred_key
except Exception:
    pass

# Verify
fred_key = os.environ.get("FRED_API_KEY", "")
print(f"FRED_API_KEY: {"set (" + fred_key[:4] + "...)" if fred_key else "NOT SET (will use dummy macro data)"}")


FRED_API_KEY: set (a5c3...)


## 3. Load Config & Backfill Historical Data

Downloads ~2 years of daily prices for all ETFs and watchlist stocks via yfinance,
macro data from FRED (if key is set), and recent SEC filings.

In [13]:
import yaml
import sqlite3
import inspect
import pandas as pd
from pathlib import Path

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ----- CRITICAL: Define a single DB path used by EVERYTHING -----
DB_PATH = Path(os.getcwd()) / 'rotation_system.db'

# --- Helper: patch a module's DB_PATH everywhere it matters ---
def patch_db_path(mod, new_path):
    """Overwrite a module's DB_PATH global AND fix any function
    defaults that captured the old value at import time."""
    old_path = getattr(mod, 'DB_PATH', None)
    mod.DB_PATH = new_path
    for name, obj in inspect.getmembers(mod, inspect.isfunction):
        if obj.__defaults__:
            new_defaults = []
            changed = False
            for d in obj.__defaults__:
                if isinstance(d, Path) and old_path and d == old_path:
                    new_defaults.append(new_path)
                    changed = True
                else:
                    new_defaults.append(d)
            if changed:
                obj.__defaults__ = tuple(new_defaults)
    print(f'  {mod.__name__}.DB_PATH -> {mod.DB_PATH}')

print("Config loaded. Portfolio total: ${:,.0f}".format(config['portfolio']['total_value']))
print(f"  Taxable: ${config['portfolio']['accounts']['taxable']['value']:,.0f}")
print(f"  Roth IRA: ${config['portfolio']['accounts']['roth_ira']['value']:,.0f}")
print(f"  Sector ETFs: {len(config['tickers']['sector_etfs'])}")
wl_keys = ['watchlist_biotech', 'watchlist_ai_software', 'watchlist_defense', 'watchlist_green_materials']
wl_count = sum(len(config['tickers'].get(k, [])) for k in wl_keys)
print(f"  Watchlist tickers: {wl_count}")
print(f"  DB path: {DB_PATH}")

Config loaded. Portfolio total: $144,000
  Taxable: $100,000
  Roth IRA: $44,000
  Sector ETFs: 11
  Watchlist tickers: 47
  DB path: /content/sector-rotation-system/rotation_system.db


In [ ]:
import gc
import sqlite3
import datetime as dt

# Force cleanup of any stale DB connections from prior cell runs
gc.collect()

# Verify DB is accessible before proceeding
if DB_PATH.exists():
    try:
        _test_conn = sqlite3.connect(str(DB_PATH), timeout=30)
        _test_conn.execute("PRAGMA journal_mode=WAL")
        _test_conn.execute("SELECT 1")
        _test_conn.close()
        print(f"✅ DB file verified: {DB_PATH}")
    except sqlite3.OperationalError as e:
        print(f"⚠️ DB appears locked from a prior run, removing stale file: {e}")
        DB_PATH.unlink()

import data_feeds
patch_db_path(data_feeds, DB_PATH)

# Single shared connection — eliminates competing connections that cause DB lock
conn = data_feeds.init_database(DB_PATH)

# Thresholds for incremental ingestion
PRICE_FRESHNESS_DAYS = 3   # skip price download if DB is this fresh
MIN_FILINGS_THRESHOLD = 50  # skip filings download if DB has more than this

try:
    # --- Incremental Prices ---
    print("\n📈 Checking price data...")
    latest_date = None
    try:
        row = conn.execute("SELECT MAX(date) FROM prices").fetchone()
        if row and row[0]:
            latest_date = row[0]
    except Exception:
        pass

    today = dt.date.today()
    if latest_date and (today - dt.date.fromisoformat(latest_date)).days <= PRICE_FRESHNESS_DAYS:
        print(f"✅ Prices are current (latest: {latest_date}). Skipping download.")
        prices_summary = {"skipped": True}
    else:
        if latest_date:
            price_start = (dt.date.fromisoformat(latest_date) + dt.timedelta(days=1)).isoformat()
            print(f"📈 Fetching incremental prices from {price_start} to today...")
        else:
            price_start = None
            print("📈 First run — fetching full 2-year price history...")
        prices = data_feeds.fetch_prices(config, start_date=price_start)
        prices, price_warnings = data_feeds.validate_prices(prices, config)
        data_feeds.store_prices(conn, prices)
        prices_summary = {
            "rows": len(prices),
            "tickers": int(prices["ticker"].nunique()) if not prices.empty else 0,
        }
        if price_warnings:
            print(f"  ⚠️ {len(price_warnings)} price warnings")
        print(f"✅ Prices stored: {prices_summary}")

    # --- Macro (always refresh — fast ~5s) ---
    print("\n📊 Fetching macro data from FRED...")
    macro = data_feeds.fetch_macro_data(config)
    macro, macro_warnings = data_feeds.validate_macro(macro)
    data_feeds.store_macro(conn, macro)
    macro_summary = {
        "rows": len(macro),
        "series": int(macro["series_id"].nunique()) if not macro.empty else 0,
    }
    print(f"✅ Macro stored: {macro_summary}")

    # --- SEC Filings (skip if already populated) ---
    print("\n📄 Checking SEC filings...")
    filing_count = conn.execute("SELECT COUNT(*) FROM filings").fetchone()[0]
    if filing_count > MIN_FILINGS_THRESHOLD:
        print(f"✅ Filings already populated ({filing_count:,} rows). Skipping download.")
        print("   To force a refresh, delete the DB file and re-run this cell.")
        filings_summary = {"skipped": True, "existing": filing_count}
    else:
        print("📄 First run — fetching SEC filings (this may take 3-10 min)...")
        filings = data_feeds.fetch_all_filings(config)
        data_feeds.store_filings(conn, filings)
        filings_summary = {"count": len(filings)}
        print(f"✅ Filings stored: {filings_summary}")

finally:
    conn.close()

# --- Ticker Coverage Report ---
import pandas as pd
print(f"\n=== Ticker Coverage ===")
print(f"  DB file exists: {DB_PATH.exists()}")
if DB_PATH.exists():
    print(f"  DB file size: {DB_PATH.stat().st_size:,} bytes")
    conn2 = sqlite3.connect(str(DB_PATH))
    try:
        for table in ['prices', 'macro_data', 'filings', 'signals']:
            try:
                count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn2).iloc[0]['n']
                print(f"  {table}: {count:,} rows")
            except Exception as e:
                print(f"  {table}: ERROR - {e}")
        loaded = pd.read_sql('SELECT DISTINCT ticker FROM prices', conn2)['ticker'].tolist()
        all_expected = set()
        for key in config['tickers']:
            if isinstance(config['tickers'][key], list):
                all_expected.update(config['tickers'][key])
        missing = all_expected - set(loaded)
        print(f"\n  Tickers loaded: {len(loaded)} / {len(all_expected)} expected")
        if missing:
            print(f"  Missing tickers ({len(missing)}): {sorted(missing)}")
            print(f"  NOTE: OTC ADRs (BAESY, RNMBY, SAABF) and VIX indices")
            print(f"  often fail on yfinance. This is expected and won't")
            print(f"  affect the core ETF-based optimization.")
    finally:
        conn2.close()
else:
    print("  ERROR: DB file was NOT created! Check ingestion logs above.")


## 4. Detect Current Regime

In [ ]:
import regime_detector
patch_db_path(regime_detector, DB_PATH)

regime_result = regime_detector.run_regime_detection(config)

print("=" * 60)
print("CURRENT REGIME")
print("=" * 60)
print(f"  Regime:              {regime_result.get('dominant_regime', 'N/A')}")
print(f"  Wedge Volume %:      {regime_result.get('wedge_volume_percentile', 'N/A')}")
print(f"  Fast Shock Risk:     {regime_result.get('fast_shock_risk', 'N/A')}")
print(f"  VIX/RV Ratio:        {regime_result.get('vix_rv_ratio', 'N/A')}")
print(f"  Confirmed:           {regime_result.get('regime_confirmed', 'N/A')}")
print(f"  Consecutive Days:    {regime_result.get('consecutive_days_in_regime', 'N/A')}")

## 5. Run CVaR Optimization

Runs the full CVaR optimizer with Ledoit-Wolf shrinkage and
Longin-Solnik tail correlations. Dollar amounts are split between
Taxable ($100K) and Roth IRA ($44K) using tax-location rules.

In [ ]:
import portfolio_optimizer
patch_db_path(portfolio_optimizer, DB_PATH)

regime = regime_result.get('dominant_regime', 'offense')
opt_result = portfolio_optimizer.run_portfolio_optimization(cfg=config, regime=regime)

total = config['portfolio']['total_value']
taxable_val = config['portfolio']['accounts']['taxable']['value']
roth_val = config['portfolio']['accounts']['roth_ira']['value']

print("=" * 80)
print(f"CVaR-OPTIMIZED ALLOCATION (Regime: {regime.upper()})")
print(f"Portfolio: ${total:,.0f} = ${taxable_val:,.0f} Taxable + ${roth_val:,.0f} Roth IRA")
print("=" * 80)

positions = opt_result.get('positions', {})
if positions:
    print(f"\n{'Ticker':<8} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  Account")
    print("-" * 75)
    for ticker, info in sorted(positions.items(), key=lambda x: -x[1].get('pct', 0)):
        pct = info.get('pct', 0)
        total_d = info.get('total_dollars', pct / 100.0 * total)
        tax_d = info.get('taxable_dollars', 0)
        roth_d = info.get('roth_dollars', 0)
        acct = info.get('account', 'N/A')
        print(f"{ticker:<8} {pct:>6.1f}% {total_d:>10,.0f} {tax_d:>10,.0f} {roth_d:>10,.0f}  {acct}")

    # Summary by account
    tax_total = sum(info.get('taxable_dollars', 0) for info in positions.values())
    roth_total = sum(info.get('roth_dollars', 0) for info in positions.values())
    print("-" * 75)
    print(f"{'TOTAL':<8} {'100%':>7} {total:>10,.0f} {tax_total:>10,.0f} {roth_total:>10,.0f}")
    print(f"\nAccount utilization: Taxable {tax_total/taxable_val:.0%} | Roth {roth_total/roth_val:.0%}")
else:
    print("No positions returned. Check optimizer logs above.")

## 6. Walk-Forward Backtest

Simulates the system over the historical period, rebalancing monthly
based on the regime detected at each rebalance date.

In [ ]:
import numpy as np
import json as json_module

conn = sqlite3.connect(str(DB_PATH))

# Load SPY as benchmark
spy = pd.read_sql(
    "SELECT date, close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_returns = spy['close'].pct_change().dropna()

# FIX: regime_detector stores signal_type='regime_state', not 'regime'
signals = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime_state' ORDER BY date",
    conn, parse_dates=['date']
)
conn.close()

print(f"SPY returns: {len(spy_returns)} days")
print(f"Regime signals: {len(signals)} rows")

if len(signals) > 0 and len(spy_returns) > 0:
    signals['regime'] = signals['signal_data'].apply(
        lambda x: json_module.loads(x).get('dominant_regime', 'offense') if isinstance(x, str) else 'offense'
    )

    regime_exposure = {'offense': 0.75, 'defense': 0.35, 'panic': 0.05}

    signals = signals.set_index('date')
    merged = spy_returns.to_frame('spy_ret').join(signals['regime'], how='left')
    merged['regime'] = merged['regime'].ffill().fillna('offense')
    merged['exposure'] = merged['regime'].map(regime_exposure)
    merged['strategy_ret'] = merged['spy_ret'] * merged['exposure']

    merged['spy_cum'] = (1 + merged['spy_ret']).cumprod()
    merged['strategy_cum'] = (1 + merged['strategy_ret']).cumprod()

    days = len(merged)
    ann_factor = 252 / days if days > 0 else 1

    spy_total = merged['spy_cum'].iloc[-1] - 1
    strat_total = merged['strategy_cum'].iloc[-1] - 1

    spy_ann = (1 + spy_total) ** ann_factor - 1
    strat_ann = (1 + strat_total) ** ann_factor - 1

    spy_vol = merged['spy_ret'].std() * np.sqrt(252)
    strat_vol = merged['strategy_ret'].std() * np.sqrt(252)

    spy_sharpe = spy_ann / spy_vol if spy_vol > 0 else 0
    strat_sharpe = strat_ann / strat_vol if strat_vol > 0 else 0

    def max_dd(cum_series):
        peak = cum_series.cummax()
        dd = (cum_series - peak) / peak
        return dd.min()

    spy_mdd = max_dd(merged['spy_cum'])
    strat_mdd = max_dd(merged['strategy_cum'])

    mclean_decay = config['factor_model']['mclean_pontiff_decay']
    alpha_raw = strat_ann - spy_ann
    alpha_adj = alpha_raw * mclean_decay
    mp_label = config['backtest']['mclean_pontiff_label']

    # Dollar impact on the $144K portfolio
    dollar_gain_spy = spy_total * total
    dollar_gain_strat = strat_total * total

    print("=" * 70)
    print("WALK-FORWARD BACKTEST RESULTS")
    print("=" * 70)
    period_start = merged.index[0].strftime('%Y-%m-%d')
    period_end = merged.index[-1].strftime('%Y-%m-%d')
    print(f"Period: {period_start} to {period_end} ({days} trading days)")
    print()
    print(f"{'Metric':<30} {'SPY (B&H)':>15} {'Strategy':>15}")
    print("-" * 60)
    print(f"{'Total Return':<30} {spy_total:>14.1%} {strat_total:>14.1%}")
    print(f"{'Annualized Return':<30} {spy_ann:>14.1%} {strat_ann:>14.1%}")
    print(f"{'Annualized Volatility':<30} {spy_vol:>14.1%} {strat_vol:>14.1%}")
    print(f"{'Sharpe Ratio':<30} {spy_sharpe:>14.2f} {strat_sharpe:>14.2f}")
    print(f"{'Max Drawdown':<30} {spy_mdd:>14.1%} {strat_mdd:>14.1%}")
    print(f"{'Dollar Gain ($144K port.)':<30} {dollar_gain_spy:>13,.0f} {dollar_gain_strat:>13,.0f}")
    print()
    print(f"Raw Alpha vs SPY:      {alpha_raw:>+.2%}")
    print(f"Adjusted Alpha (x{mclean_decay}): {alpha_adj:>+.2%}")
    print(f"  {mp_label}")

    # Regime distribution
    print(f"\nRegime distribution:")
    regime_counts = merged['regime'].value_counts()
    for r, ct in regime_counts.items():
        print(f"  {r}: {ct} days ({ct/days:.0%})")
else:
    print("Not enough data for backtest. Run data ingestion first.")
    print(f"  Debug: SPY returns = {len(spy_returns)}, regime signals = {len(signals)}")

## 7. Stock Screener — Top Picks

In [ ]:
import stock_screener
patch_db_path(stock_screener, DB_PATH)

screener_result = stock_screener.run_stock_screener(cfg=config, regime=regime)

print("=" * 60)
print("THEMATIC WATCHLIST SCORES")
print("=" * 60)

# FIX: The screener returns 'watchlist_scores', not 'watchlists'
watchlists = screener_result.get('watchlist_scores', {})
for name, data in watchlists.items():
    if data:
        print(f"
--- {name.upper()} ---")
        # data may be a list of dicts (serialized from DataFrame)
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif hasattr(data, 'to_string'):
            df = data
        else:
            print(f"  (unexpected format: {type(data).__name__})")
            continue
        if not df.empty:
            # Show the most useful columns
            show_cols = [c for c in ['ticker', 'composite_score', 'momentum_score',
                                      'quality_score', 'value_score', 'size_score',
                                      'valuation_label', 'signal'] if c in df.columns]
            if show_cols:
                print(df[show_cols].head(10).to_string(index=False))
            else:
                print(df.head(10).to_string(index=False))
        else:
            print("  (empty)")

# Show ENTRY/EXIT signals grouped by source
signals_out = screener_result.get('signals', {})
if signals_out:
    print(f"
{'='*60}")
    print("ENTRY / EXIT SIGNALS")
    print(f"{'='*60}")
    for sig_type in ['entry', 'exit']:
        sigs = signals_out.get(sig_type, [])
        if not sigs:
            print(f"
--- {sig_type.upper()}: none ---")
            continue
        print(f"
--- {sig_type.upper()} ---")
        # Group by source so both watchlist and sector_screen signals are visible
        watchlist_sigs = [s for s in sigs if s.get('source', 'watchlist') != 'sector_screen']
        sector_sigs = [s for s in sigs if s.get('source') == 'sector_screen']
        if watchlist_sigs:
            print("  [Watchlist]")
            for s in watchlist_sigs[:10]:
                print(f"    {s.get('ticker', '?')} ({s.get('watchlist', '')}): {s.get('reason', '')}")
        if sector_sigs:
            print("  [Sector Screen]")
            for s in sector_sigs[:10]:
                print(f"    {s.get('ticker', '?')} ({s.get('watchlist', '')}): {s.get('reason', '')}")

# --- Sector Top Picks (cross-sector, from overweight sectors) ---
print(f"
{'='*60}")
print("SECTOR TOP PICKS (best stocks from overweight sectors)")
print(f"{'='*60}")
stp_data = screener_result.get('sector_top_picks', [])
if stp_data:
    stp_df = pd.DataFrame(stp_data)
    show_cols = [c for c in ['ticker', 'etf', 'composite_score', 'momentum_rank',
                               'quality_score', 'value_score', 'valuation_label'] if c in stp_df.columns]
    print(stp_df[show_cols].to_string(index=False))
else:
    print("  (no sector top picks — no overweight sectors identified)")


## 8. Run Full Monitor (Executive Summary)

In [ ]:
# Run the full monitor pipeline and display the executive summary.
# --no-deliver skips Telegram/email/Sheets delivery (not configured in Colab).
# --db explicitly passes our DB path to avoid any path mismatch.
!python monitor.py --no-deliver --db "{DB_PATH}" 2>&1 | head -120

## 9. Download Database

Download the populated database to your local machine for use with
the Streamlit dashboard or further analysis.

In [ ]:
from google.colab import files

# Download the populated database
files.download(str(DB_PATH))
print("Database downloaded. Place it in the repo root for the dashboard.")